# Quick SLM — 04 · SFT corpus

Builds the supervised corpus that turns the base checkpoint into an agent. Four stages,
each resumable, each implemented in `quick_slm_trainer.sft`:

    plan_requests -> run_generation -> load_and_validate -> dedup -> pack

The design doc is `training/SFT_README.md`. Three decisions here depart from it.

**The teacher emits JSON, not the template.** The doc has Gemma write out the canonical
ChatML envelope and a validator check that it parses. Asking a model to reproduce a surface
form exactly makes it responsible for whitespace it has no reason to care about, and it
means the corpus's surface form is whatever the teacher emitted while the inference
server's is whatever `template.py` emits. Instead the teacher supplies content, and
`template.render` produces the exact bytes. The teacher cannot get the format wrong because
it never sees the format, and validation becomes structural: a parsed call checked against
the schema it was generated against.

**Examples are bin-packed whole, not concatenated.** Concatenation splits an example across
a window boundary, cutting an assistant turn mid-`<response>` and scoring the truncated
half. Structured output is the thing this stage exists to teach.

**Category 3 is generated and accepted in counterfactual pairs.** Two prompts identical
except at one field of `<state>`, an oracle that computes the correct call from state alone,
and a rule that drops both branches unless both match. The same user turn appears twice with
two different correct answers, so the user turn predicts nothing about which call is right
and `<state>` is the only feature that does. The filter this replaces accepted any `<think>`
block containing the words "state" and "memory", which selected for narration rather than
behaviour and could not have supported the paper's central claim. See `sft/conflict.py`.

Generation is the expensive part: roughly 10 to 15 H100 hours for 80 k examples. Everything
after it runs on CPU in minutes, so a filter tightened later costs nothing to re-apply. Only
this notebook needs a GPU; training is notebook 05.

## 1 · Drive and GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

## 2 · Framework

In [ ]:
EXTRAS = "sft"

# Framework and data live on Drive (no GitHub). Upload the repo folder
# (with pyproject.toml, README.md, and framework/) into DRIVE_ROOT/code once.
# Every notebook installs it from there.
DRIVE_ROOT = "/content/drive/MyDrive/quick-slm"

import subprocess, sys, importlib
from pathlib import Path

root = Path(DRIVE_ROOT)
candidates = [root / "code", root, root / "quick-slm"]
REPO_DIR = next((p for p in candidates if (p / "pyproject.toml").exists()), None)
if REPO_DIR is None:
    looked = "\n  ".join(str(p) for p in candidates)
    raise RuntimeError(
        "No pyproject.toml found on Drive. Upload the repo (pyproject.toml, "
        f"README.md, and framework/) to {root / 'code'}, then re-run.\nLooked in:\n  "
        + looked
    )

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[{EXTRAS}]"], check=True)

# The editable install lands its import hook in site-packages as a .pth file, and
# a .pth is only executed at interpreter startup. This kernel is already running,
# so it never sees the hook and `import v1.quick_slm_trainer` fails until a restart.
# Put framework/ on the path directly and drop the finder caches, so
# the import resolves in this session with no "Restart runtime" step.
framework_dir = REPO_DIR / "framework"
if str(framework_dir) not in sys.path:
    sys.path.insert(0, str(framework_dir))
importlib.invalidate_caches()

import v1.quick_slm_trainer as q
print("quick-slm-trainer", q.__version__, "from", Path(q.__file__).parent)
# Refuse to run outside the support window this training version declares in
# training/v1/framework.json. A framework newer than v1's window would not fail
# loudly, it would build a different corpus and the difference would surface as
# unexplained numbers weeks later. If this raises, install the archived build it
# names rather than editing this cell.
if hasattr(q, "require_framework"):
    q.require_framework("v1", REPO_DIR)
else:
    raise RuntimeError(
        "quick-slm-trainer " + q.__version__ + " predates the support-window check; "
        "training v1 requires >=1.0. See SUPPORT.md."
    )


## 3 · Plan

The whole request plan is built first, from a seeded RNG, before a single token is
generated. Composition is knowable in advance, and each request has a stable id, so a run
killed at 60 % resumes at 60 % rather than re-planning.

`target_examples` is the *raw* count. `training/SFT_README.md` budgets a 25 to 35 % total
reject rate across validation and dedup. Category 3 is planned in whole pairs, so its count
is always even, and it is generated at two prompts per pair.

In [ ]:
from v1.quick_slm_trainer import Layout, sft_v1
from v1.quick_slm_trainer.sft import describe_plan, plan_requests

layout = Layout().mkdirs_sft()
cfg = sft_v1()

requests = plan_requests(cfg.sft)
print(describe_plan(requests))

paired = [r for r in requests if r.is_paired]
print(f'\ncounterfactual branches: {len(paired):,}  ({len(paired) // 2:,} pairs)')

# Two capacity caps shape this plan, one per half of the corpus, and both keep the
# teacher off duplicates dedup would later delete at full price. The paired category
# is held to what its spec table can distinguish (reported below). The unpaired
# categories are held to `max_examples_per_seed` examples per distinct seed, because
# their scenarios come from hand-written pools of tens of topics, and planning
# thousands per cell only makes the teacher rephrase a seed it already has.
print(f'\nunpaired cap: max_examples_per_seed = {cfg.sft.max_examples_per_seed} '
      f'(per distinct seed, per domain/subtype cell)')


### The counterfactual specs, before anything is generated

Two properties have to hold, and both are cheap to check now and impossible to recover
later.

**Every spec produces a decisive flip.** If changing the field named by `decisive_path` does
not change the correct call, the pair cannot separate grounding from imitation, and the
*spec* is at fault, not the sample. `check_spec` rebuilds each one two dozen times over
different random instances and raises on the first failure.

**Train and eval share nothing.** The paper's evaluation is this same construction, so the
partition is at the spec level and made before generation: no tool family, no decisive path,
and no state schema appears on both sides, and one family and one whole domain are held out
entirely. Partitioning at the sample level would leak, because the two branches of a pair
differ in a single field.

In [ ]:
import random

from v1.quick_slm_trainer.sft.conflict import assert_no_leakage, build_pair, check_spec
from v1.quick_slm_trainer.sft.conflict_specs import EVAL_SPECS, TRAIN_SPECS
from v1.quick_slm_trainer.sft.oracles import canonical_call

for spec in (*TRAIN_SPECS, *EVAL_SPECS):
    check_spec(spec)          # raises SpecRejected if the flip is not decisive

assert_no_leakage(TRAIN_SPECS, EVAL_SPECS)

print(f'{len(TRAIN_SPECS)} train specs, {len(EVAL_SPECS)} held out for eval\n')
print(f'{"spec":<20s} {"family":<16s} {"decisive path":<26s} {"call(a)":<16s} call(b)')
print('-' * 96)
for spec in TRAIN_SPECS:
    p = build_pair(spec, random.Random(0))
    print(f'{spec.id:<20s} {spec.family:<16s} {spec.decisive_path:<26s} '
          f'{canonical_call(p.a.call)[0]:<16s} {canonical_call(p.b.call)[0]}')

print(f'\nheld out: families {sorted({s.family for s in EVAL_SPECS})}, '
      f'domains {sorted({s.domain for s in EVAL_SPECS} - {s.domain for s in TRAIN_SPECS})}')

In [ ]:
# What the spec table can actually distinguish, computed from the same fingerprint
# the dedup pass uses: re-randomised ids are stripped and `answer`'s prose is
# excluded, so a spec's capacity is the product of its own small choice lists.
# Planning past it buys duplicates at teacher price, so `sft_v1` caps the paired
# plan here rather than discovering the waste after ten hours of GPU.
from v1.quick_slm_trainer.sft.conflict import describe_capacity, recommended_conflict_pairs

print(describe_capacity(TRAIN_SPECS, len(paired) // 2, trials=6000))
print('\nEVAL (the paper reports its state-grounding number on these held-out specs):')
print(describe_capacity(EVAL_SPECS, 500, trials=6000))
print(f'\nsft_v1 caps the paired plan at {cfg.sft.conflict_max_pairs} pairs, '
      f'from recommended_conflict_pairs() = {recommended_conflict_pairs()}.')

Look at one pair. The two prompts must differ on exactly one line, inside `<state>`. Any
other difference, an id or a reordered key, is a cue the model can read instead of the
state, and the pair silently stops measuring anything.

In [ ]:
import difflib

from v1.quick_slm_trainer.sft.conflict import prompt_block
from v1.quick_slm_trainer.sft.conflict_specs import ALL_SPECS

# The sharpest spec: both branches call `buy`, and only the quantity differs. Guessing the
# tool from the user turn is not enough; the argument has to come from <state>.
pair = build_pair(ALL_SPECS['inventory_amount'], random.Random(7))
a, b = prompt_block(pair, 'a'), prompt_block(pair, 'b')

changed = [l for l in difflib.unified_diff(a.split('\n'), b.split('\n'), lineterm='')
           if l.startswith(('+', '-')) and not l.startswith(('+++', '---'))]
assert len(changed) == 2 and all('<state>' in l for l in changed), changed

print('the only difference between the two prompts:')
for l in changed:
    print(' ', l[:150])
print(f'\noracle a: {pair.a.call}')
print(f'oracle b: {pair.b.call}')
print(f'\n--- branch a, exactly what the student will condition on ---\n{a}')

And one unpaired prompt, from the trap category, before spending GPU hours on eighty
thousand of them.

## 4 · Teacher

Gemma 4 31 B, loaded from Google's QAT Q4 checkpoint (`-qat-q4_0-unquantized`) in 4-bit
nf4. bf16 is banned here: the full 31 B in bf16 is roughly 62 GB before the KV cache, and a
batch of 32 at 2048 tokens on top of that OOMs a G4's 96 GB. The Q4 weights land near 18 GB
and leave the rest of the card for KV headroom. Quantization-aware training means the Q4
weights were tuned for this, so the signal loss over bf16 is small. A single dense teacher
with conservative sampling gives a 103 M student a sharper signal than a mixture would: the
student's capacity to learn a behaviour is bounded by how consistent the demonstrations are.

`load_teacher` sets left padding. A decoder-only model batched with right padding generates
from the pad tokens and returns nonsense for every sequence shorter than the longest in the
batch, silently.

If `cfg.sft.teacher_model` is unavailable, the pipeline is teacher-agnostic: only the name
changes.

In [ ]:
import os
# Reduce allocator fragmentation before the first CUDA allocation, as notebooks 02 and
# 05 do. Generation runs the categories back to back on one card, and the reserved pool
# a category leaves behind is what OOMs the next; expandable segments let the allocator
# grow into that space instead of failing beside it. `run_generation` also empties the
# cache at each category boundary and halves any batch that still OOMs, but setting this
# first is what keeps those from being needed in the common case.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from v1.quick_slm_trainer.sft import generate as G

teacher, teacher_tok = G.load_teacher(cfg.sft)
print('teacher loaded:', cfg.sft.teacher_model)


In [ ]:
# Pre-flight: generate a handful across every category and read what the teacher
# actually did, before committing a day of GPU. It confirms three things the full
# run assumes: JSON validity holds now that repetition_penalty is 1.0; the chat
# template injects no reasoning trace whose braces `extract_json` would parse in
# place of the example (Gemma 4 leaves thinking off unless the prompt asks for it,
# but the template is the teacher's, not ours); and the batch fits in memory.
flight = G.preflight(teacher, teacher_tok, cfg.sft, requests, n=20)
print(G.describe_preflight(flight))

# Read one raw reply end to end. A corpus nobody looked at is a corpus nobody knows.
print('\n--- one raw teacher reply, verbatim ---')
print(flight[0].raw[:1200])

In [ ]:
# Distinct turns per seed, on the cell dedup collapses hardest. This is the pre-flight
# form of what `max_examples_per_seed` caps, and the measurement notebook 04a section 6
# previews by hand: it replays each seed of single_stage/world/direct a handful of times
# and counts the distinct (user, calls) the teacher actually produces. A yield well under
# the cap means the cap is still loose and teacher time is going on duplicates dedup will
# delete; a yield near it means the cap is set about right. Opt-in, because it costs a few
# hundred generations rather than the pre-flight's twenty.
MEASURE_SEED_YIELD = False

if MEASURE_SEED_YIELD:
    sy = G.distinct_turns_per_seed(
        teacher, teacher_tok, cfg.sft, requests,
        category='single_stage', domain='world', subtype='direct', per_seed=20,
    )
    print(G.describe_seed_yield(sy))
    print(f'\ncap in force: max_examples_per_seed = {cfg.sft.max_examples_per_seed}')
else:
    print('MEASURE_SEED_YIELD is False; skipping the distinct-turns-per-seed probe.')
    print(f'cap in force: max_examples_per_seed = {cfg.sft.max_examples_per_seed}.')
    print('Set it True to spend a few hundred generations checking the teacher yield on')
    print('single_stage/world/direct, the cell dedup collapses hardest, against the cap.')


## 5 · Generate

Appends one JSONL shard per category under `sft/raw/`. Raw teacher text is stored, not
parsed output, because parsing is cheap and re-runnable while generation is not.

Re-run this cell after a disconnect. It reads the ids already on disk and skips them.

In [ ]:
written = G.run_generation(
    layout=layout, cfg=cfg.sft, model=teacher, tok=teacher_tok, requests=requests,
)
print('\nnew records:', written)

In [ ]:
# Free the teacher before the CPU stages. Nothing below needs a GPU.
import gc, torch

del teacher
gc.collect()
torch.cuda.empty_cache()

## 6 · Validate

Every filter rejects rather than repairs. A repaired example is one the teacher got wrong
and the pipeline guessed at, and the student learns the guess as readily as the intent.

Category 3 is judged pairwise. Both branches must match their oracle; if the teacher gets
one right and the other wrong, both are dropped, because the correct branch on its own is
indistinguishable from a lucky pattern match. `<think>` is checked for being non-empty and
free of leaked markup, and for nothing else, in every category alike.

The oracle is recomputed here from `conflict_specs`, not read out of the shard. The shard
holds the inputs; the code owns the ground truth. Correcting an oracle therefore costs a
revalidation pass, never another generation run.

In [ ]:
from v1.quick_slm_trainer.sft import load_and_validate
from v1.quick_slm_trainer.sft.corpus import category_histogram, paired_integrity, subtype_histogram

examples, stats, per_category = load_and_validate(layout, cfg.sft)

print('\n=== corpus-wide ===')
print(stats.report())
print('\naccepted by category:', category_histogram(examples))

n_pairs, broken = paired_integrity(examples)
assert not broken, f'lone branches survived validation: {broken[:5]}'
print(f'\ncounterfactual pairs accepted: {n_pairs:,} (both branches each)')

Read a few. A corpus nobody looked at is a corpus nobody knows the contents of.

In [ ]:
from v1.quick_slm_trainer.template import render

for cat in ('single_stage', 'multi_stage', 'state_memory_conflict', 'traps', 'refusals'):
    ex = next((e for e in examples if e.category == cat), None)
    if ex is None:
        print(f'--- {cat}: none survived validation ---\n')
        continue
    print('=' * 78)
    print(f'{cat} / {ex.meta.get("subtype", "")}')
    print('=' * 78)
    print(render(ex)[:1800])
    print()

## 7 · Dedup

MinHash over the user request plus the call signatures, Jaccard >= 0.85. The `<think>` block
is deliberately excluded from the fingerprint: two examples asking the same thing and
answering with the same calls are duplicates however differently the reasoning is worded.

The two branches of a counterfactual pair share a group and are compared as one document, so
a pair is kept or dropped whole. An ungrouped pass could keep one branch and drop its twin,
which re-admits the isolated sample the pairing exists to exclude.

In [ ]:
from v1.quick_slm_trainer.sft.dedup import dedup_examples

before = len(examples)
examples = dedup_examples(examples, threshold=cfg.sft.dedup_jaccard,
                          num_perm=cfg.sft.minhash_perms, progress=True)
print(f'{before:,} -> {len(examples):,}  ({100 * (before - len(examples)) / max(before, 1):.1f}% duplicate)')
print('after dedup:', category_histogram(examples))

n_pairs, broken = paired_integrity(examples)
assert not broken, f'dedup split a pair: {broken[:5]}'
print(f'counterfactual pairs surviving: {n_pairs:,}')

## 8 · Split and pack

The validation split is held out at the **example** level, before packing: splitting after
would put two halves of one window on both sides of the boundary. Groups move whole, so a
counterfactual pair never has one branch in train and its twin in val, which would be a
validation example the model had all but memorised.

Each window holds as many complete examples as fit and is padded to `ctx` with EOS at mask
0. Padding costs nothing at training time, since padded positions are never scored and
causal attention means no real token can attend forward into them.

In [ ]:
from v1.quick_slm_trainer.sft.pack import pack_split, split_examples, write_stats
from v1.quick_slm_trainer.tokenizer import load_tokenizer

tok = load_tokenizer(layout.tokenizer_dir, patch=False)
cfg.sft.ctx = cfg.data.ctx

train_examples, val_examples = split_examples(examples, val_fraction=cfg.sft.val_fraction)
print(f'train {len(train_examples):,}   val {len(val_examples):,}')

for name, split in (('train', train_examples), ('val', val_examples)):
    _, broken = paired_integrity(split)
    assert not broken, f'{name} split contains a lone branch: {broken[:5]}'
print('every counterfactual pair landed on one side of the split\n')

train_stats = pack_split(layout, 'train', train_examples, tok, cfg.sft)
print()
val_stats = pack_split(layout, 'val', val_examples, tok, cfg.sft)

### Corpus statistics

Written to `sft/corpus_stats.json`. The paper's data section reads this file.

In [ ]:
path = write_stats(layout, {
    'config': cfg.sft.to_dict(),
    'validation': {'overall': stats.to_dict(),
                   'by_category': {k: v.to_dict() for k, v in per_category.items()}},
    'after_dedup': {'examples': len(examples),
                    'by_category': category_histogram(examples),
                    'by_subtype': subtype_histogram(examples)},
    'pack': {'train': train_stats.to_dict(), 'val': val_stats.to_dict()},
})

print('wrote', path)
print(f"\npacked train tokens : {train_stats.total_tokens:,}")
print(f"packed val tokens   : {val_stats.total_tokens:,}")
print(f"scored fraction     : {100 * train_stats.scored_fraction:.1f}%")